## Practical 4

# Setup and Loading Data

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Load the cleaned dataset
df = pd.read_csv('../data/heart_clean.csv')

display(df.head())

,id,age,sex,dataset,trestbps,chol,fbs,thalch,exang,oldpeak,...,num,cp_atypical angina,cp_non-anginal,cp_typical angina,thal_normal,thal_reversable defect,slope_flat,slope_upsloping,restecg_normal,restecg_st-t abnormality
0,-1.730169,1.007386,Male,Cleveland,0.698041,0.311021,True,0.495698,False,1.349421,...,0,False,False,True,False,False,False,False,False,False
1,-1.726404,1.432034,Male,Cleveland,1.511761,0.797713,False,-1.175955,True,0.589832,...,2,False,False,False,True,False,True,False,False,False
2,-1.722639,1.432034,Male,Cleveland,-0.658158,0.274289,False,-0.340128,True,1.634267,...,1,False,False,False,False,True,True,False,False,False
3,-1.718873,-1.752828,Male,Cleveland,-0.115679,0.467130,False,1.968345,False,2.488805,...,0,False,True,False,True,False,False,False,True,False
4,-1.715108,-1.328180,Female,Cleveland,-0.115679,0.044717,False,1.371326,False,0.494884,...,0,True,False,False,True,False,False,True,False,False


# Define Features (X) and Target (y)

In [3]:
# Drop irrelevant columns to isolate our Features (X)
X = df.drop(columns=['num', 'id', 'dataset'], errors='ignore')

X = pd.get_dummies(X, drop_first=True)

y = (df['num'] > 0).astype(int)

print("Features (X) shape:", X.shape)
print("Target (y) shape:", y.shape)

Features (X) shape: (920, 18)
Target (y) shape: (920,)


# Train-Test Split & Feature Scaling

In [4]:
# Split dataset: 75% for training, 25% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data successfully split and scaled!")
print("Number of training records:", X_train_scaled.shape[0])
print("Number of testing records:", X_test_scaled.shape[0])

Data successfully split and scaled!
Number of training records: 690
Number of testing records: 230


# Analyze the Effect of 'k'

In [5]:
# Performance Analysis: Testing different values of k
print("Analyzing different values of k:\n")

best_k = 1
best_acc = 0

for k in range(1, 12, 2): # Using odd numbers prevents voting ties
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    pred = knn.predict(X_test_scaled)
    acc = accuracy_score(y_test, pred)
    
    print(f"k = {k:2d} | Accuracy = {acc:.4f}")
    
    # Keep track of the best performing k
    if acc > best_acc:
        best_acc = acc
        best_k = k

print(f"\nOptimal value found: k = {best_k} (Accuracy: {best_acc:.4f})")

Analyzing different values of k:

k =  1 | Accuracy = 0.7609
k =  3 | Accuracy = 0.8130
k =  5 | Accuracy = 0.8000
k =  7 | Accuracy = 0.8000
k =  9 | Accuracy = 0.8043
k = 11 | Accuracy = 0.8174

Optimal value found: k = 11 (Accuracy: 0.8174)


# Final Model Evaluation

In [6]:
final_knn = KNeighborsClassifier(n_neighbors=best_k)
final_knn.fit(X_train_scaled, y_train)
final_pred = final_knn.predict(X_test_scaled)

print(f"--- Final Evaluation (k={best_k}) ---")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, final_pred))

print("\nClassification Report:")
print(classification_report(y_test, final_pred))

--- Final Evaluation (k=11) ---

Confusion Matrix:
[[ 76  20]
 [ 22 112]]

Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.79      0.78        96
           1       0.85      0.84      0.84       134

    accuracy                           0.82       230
   macro avg       0.81      0.81      0.81       230
weighted avg       0.82      0.82      0.82       230

